In [2]:
import numpy as np
import tensorflow as tf

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [5]:
with open('shakespeare_complete_works.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [6]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokenizer.fit_on_texts([text])

In [7]:
full_sequence = tokenizer.texts_to_sequences([text])[0]

In [8]:
window_size = 30
input_sequence = []

In [9]:
# using a sliding window
for i in range(len(full_sequence) - window_size):
    input_sequence.append(full_sequence[i : i + window_size + 1])
input_sequences = np.array(input_sequence)

In [10]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1] # maintained as sparse integers

In [11]:
model = Sequential([Embedding(input_dim=vocab_size, output_dim=100),
                    LSTM(150),
                    Dense(vocab_size, activation='softmax')])

In [12]:
# compiling with sparse_categorical_crossentropy to save memory on such a big dataset
model.compile(loss = 'sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [28]:
def sample_with_temperature(predictions, temperature = 1.0):
    # higher temp -> more creative (random), lower temp -> more confident
    predictions = np.asarray(predictions).astype('float64')

    predictions[0] = 0.0
    predictions[1] = 0.0

    predictions = np.log(predictions + 1e-7) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, predictions, 1)
    return np.argmax(probas)



In [29]:
def generate_text(seed_text, next_words, temperature = 1.0):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=window_size, padding = 'pre')

        predicted_prob = model.predict(token_list, verbose=0)[0]
        predicted_idx = sample_with_temperature(predicted_prob, temperature)

        output_word = tokenizer.index_word.get(predicted_idx, f"<MISSING_{predicted_idx}>")
        seed_text += " " + output_word

    return seed_text

In [13]:
model.fit(X, y, epochs = 25, batch_size = 256) # not trained yet

Epoch 1/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 49s 13ms/step - accuracy: 0.0714 - loss: 6.2963
Epoch 2/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - accuracy: 0.1062 - loss: 5.7722
Epoch 3/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1179 - loss: 5.5426
Epoch 4/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - accuracy: 0.1255 - loss: 5.3835
Epoch 5/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1317 - loss: 5.2543
Epoch 6/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - accuracy: 0.1366 - loss: 5.1420
Epoch 7/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1414 - loss: 5.0406
Epoch 8/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - accuracy: 0.1456 - loss: 4.9489
Epoch 9/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1497 - loss: 4.8647
Epoch 10/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1534 - loss: 4.7876
Epoch 11/25
3370/3370 ━━━━━━━━━━━━━━━━━━━━ 45s 13ms/step - accuracy: 0.1575 - loss: 4.7154
Epoch 12

In [33]:
print(generate_text("To be or not be, that is the question", 30, temperature=0.8))

To be or not be, that is the question that see some shepherds ariel this wide air cooling a great without a week will turn his neck to the grecian end and observe the glory way and like the
